# mag2flux

Converts photometric magnitudes into fluxes using the Filter Profile Service of the Spanish Virtual Observatory service (https://svo2.cab.inta-csic.es/theory/fps/).

**Input:** name, coordinates, filter, magnitude, uncertainty in the magnitude and extinction Av.
**Output:** name, coordinates, filter identification, filter effective wavelength, flux, uncertainty in the flux, dereddened flux, uncertainty in the dereddened flux. All fluxes and their associated errors are in erg/cm2/s/A.

Name, RA, Dec and Av may be left empty. An empty value is written as `""` or as `"--"`,
and the table shows `--` in its place.

**Requirements:** `requests` and `astropy`, both already present in Colab. Locally,
`pip install requests astropy`.

In [ ]:
# 
# %pip install "astropy" "requests"
#

## 1. Settings

In [ ]:
import time
import warnings 
import xml.etree.ElementTree as ET

import requests
from astropy.io import ascii
from astropy.table import MaskedColumn, Table
from astropy.units import UnitsWarning

warnings.filterwarnings("ignore", category=UnitsWarning)

BASE_URL = "https://svo2.cab.inta-csic.es/theory/vosa/mag2flux1.php"

# Courtesy: the SVO logs should show this tool rather than an anonymous "python"
USER_AGENT = "mag2flux/1.0 (SVO)"

TIMEOUT = 30    # seconds before giving up on a silent server
PAUSE = 0.2     # seconds between requests in a batch, to be gentle on the server

# Ways of writing "there is no value here"
MISSING_MARKS = {"", "-", "--", "n/a", "na", "none", "null"}

# Fields that may be left empty
OPTIONAL_COLUMNS = ("name", "ra", "dec", "av")

# Columns that identify the object.
IDENTITY_COLUMNS = ("name", "ra", "dec")

# Columns that arrive as text from the VOTable and are wanted as numbers
NUMERIC_COLUMNS = ("leff", "mag", "emag", "av", "flux", "eflux", "dflux", "deflux")

# Order of the columns in the output table
OUTPUT_COLUMNS = ("name", "ra", "dec", "fil", "leff", "mag", "emag", "av",
                  "flux", "eflux", "dflux", "deflux")

## 2. One query

# `query_svo` sends the request and returns **every** column in a dictionary.
# `mag_to_flux` builds on it and returns only the four fluxes.

def query_svo(filter_id, mag, emag, av):
    """Query the service and return every column as a dictionary of strings."""
    url = f"{BASE_URL}?filter={filter_id}&mag={mag}&emag={emag}&av={av}"
    response = requests.get(url, timeout=TIMEOUT, headers={"User-Agent": USER_AGENT})
    root = ET.fromstring(response.text)

    # endswith is used because the VOTable carries a namespace, so the tag arrives as
    # "{http://www.ivoa.net/xml/VOTable/v1.1}INFO"
    for element in root.iter():
        if element.tag.endswith("INFO") and element.get("name") == "QUERY_STATUS":
            if element.get("value") != "OK":
                raise RuntimeError(
                    f"The SVO answered {element.get('value')}: {(element.text or '').strip()}"
                )

    # Match values to column names instead of relying on their position, so an extra
    # column added upstream will not silently shift the results
    names = [e.get("name") for e in root.iter() if e.tag.endswith("FIELD")]
    values = [e.text for e in root.iter() if e.tag.endswith("TD")]
    return dict(zip(names, values))


def mag_to_flux(filter_id, mag, emag, av):
    """Return (flux, eflux, dflux, deflux) for one filter and one magnitude."""
    row = query_svo(filter_id, mag, emag, av)
    return float(row["flux"]), float(row["eflux"]), float(row["dflux"]), float(row["deflux"])


## 3. Empty values

# Name, RA, Dec and Av are optional. Any of them can be written as `""` or `"--"`, and both
# are read as "no value given".

# Name and coordinates are only carried through, so an empty one simply stays empty. Av is
# different: it is a parameter of the query, and the service needs a number. When no Av is
# given the query is sent with **Av = 0**, meaning no dereddening, and in that case `dflux`
# comes back equal to `flux`. The Av column is still shown as `--`, to record that the
# value was not provided rather than measured as zero.

def is_missing(value):
    """True when a field was left empty or marked as having no value."""
    if value is None:
        return True
    if isinstance(value, str) and value.strip().lower() in MISSING_MARKS:
        return True
    return False


def looks_numeric(value):
    """True when the value can be read as a number."""
    try:
        float(value)
    except (TypeError, ValueError):
        return False
    return True


## 4. Many queries at once

# `convert_many` walks a list of measurements and returns one table. A failing row is
# reported and skipped rather than bringing the batch down, and a short pause between
# requests keeps the SVO server comfortable. Both cost nothing with three filters and
# matter with three hundred.

def convert_many(observations, pause=PAUSE, verbose=True):
    """Convert a list of measurements and return them as one table.

    observations : list of (name, ra, dec, filter_id, mag, emag, av)
                   Name, coordinates and Av may be empty ("" or "--").
    """
    rows, masks = [], []

    for name, ra, dec, filter_id, mag, emag, av in observations:
        # The service needs a number for Av; an empty one means no dereddening
        av_missing = is_missing(av)
        av_used = 0 if av_missing else av

        try:
            row = query_svo(filter_id, mag, emag, av_used)
        except Exception as error:
            # One bad measurement does not stop the batch
            if verbose:
                label = "(unnamed)" if is_missing(name) else name
                print(f"Skipped {label} / {filter_id}: {error}")
            continue

        for column in NUMERIC_COLUMNS:
            row[column] = float(row[column])

        # Name and coordinates come from the input, not from the SVO response
        row["name"], row["ra"], row["dec"] = name, ra, dec
        rows.append(row)

        # What to hide behind "--" in this row
        masks.append({"name": is_missing(name), "ra": is_missing(ra),
                      "dec": is_missing(dec), "av": av_missing})

    return build_table(rows, masks)


def build_table(rows, masks):
    """Assemble the masked table, where a masked cell is displayed as --."""
    if not rows:
        return Table(names=list(OUTPUT_COLUMNS))

    table = Table(masked=True)
    for column in OUTPUT_COLUMNS:
        flags = [mask.get(column, False) for mask in masks]
        values = column_values(column, [row[column] for row in rows], flags)
        table[column] = MaskedColumn(values, mask=flags)

    return describe(table)


def column_values(column, values, flags):
    """Give a column a single consistent type, with a placeholder under the mask.

    Coordinates stay numeric when every value given is a number, and become text when any
    of them is sexagesimal, so both notations survive as they were written.
    """
    if column not in OPTIONAL_COLUMNS:
        return values

    if column in ("ra", "dec"):
        given = [value for value, flag in zip(values, flags) if not flag]
        if given and all(looks_numeric(value) for value in given):
            return [0.0 if flag else float(value) for value, flag in zip(values, flags)]

    if column == "av":
        return [0.0 if flag else float(value) for value, flag in zip(values, flags)]

    return ["" if flag else str(value) for value, flag in zip(values, flags)]


def describe(table):
    """Attach units and UCDs, so both survive into the VOTable.

    The UCDs let a VO client such as TOPCAT recognise which columns hold the identity and
    the sky position, and cross-match or plot the table without being told.
    """
    if len(table) == 0:
        return table

    table["leff"].unit = "Angstrom"
    table["mag"].unit = "mag"
    table["emag"].unit = "mag"
    for column in ("flux", "eflux", "dflux", "deflux"):
        table[column].unit = "erg / (cm2 s Angstrom)"

    table["name"].meta["ucd"] = "meta.id;meta.main"
    table["ra"].meta["ucd"] = "pos.eq.ra;meta.main"
    table["dec"].meta["ucd"] = "pos.eq.dec;meta.main"

    # Degrees are only declared when the coordinates came in as numbers; sexagesimal
    # coordinates arrive as text and are left without a unit
    for column in ("ra", "dec"):
        if table[column].dtype.kind in "fi":
            table[column].unit = "deg"

    return table


## 5. Saving the table

# The extension picks the format: `.vot` for a VOTable, anything else for CSV. Empty values
# are written as `--` in the CSV and as an empty cell in the VOTable, which is how the VO
# standard represents a missing value.

def save_table(table, path):
    """Write the table to disk. A .vot extension gives a VOTable, otherwise CSV."""
    if str(path).endswith(".vot"):
        table.write(path, format="votable", overwrite=True)
    else:
        ascii.write(table, path, format="csv", overwrite=True,
                    fill_values=[(ascii.masked, "--")])
    return path

## 2. The measurements

This is the only cell that needs changing. One line per photometric point, with name and
coordinates repeated across the filters of the same source.

| Field | Meaning | May be empty |
|---|---|---|
| `name` | name of the object | yes |
| `ra` | right ascension | yes |
| `dec` | declination | yes |
| `filter_id` | filter identifier in the SVO | no |
| `mag` | magnitude | no |
| `emag` | magnitude error | no |
| `av` | visual extinction | yes, queried as 0 |

Coordinates accept decimal degrees (`79.57`) and sexagesimal notation (`"04 55 52.5"`,
in quotes). Empty fields are written as `""` or `"--"`.

In [ ]:
observations = [
    ("AR Aur", 79.57, 33.73, "2MASS/2MASS.J",  6.19, 0.019, 0.0007),
    ("AR Aur", 79.57, 33.73, "2MASS/2MASS.H",  14.6, 0.12,  2),
    ("AR Aur", 79.57, 33.73, "2MASS/2MASS.Ks", 14.3, 0.14,  2),
    ("V 1234", 85.12, -2.41, "2MASS/2MASS.H",  12.7, 0.06,  "--"),   # no extinction given
    ("", "", "", "2MASS/2MASS.J",  15.2, 0.15,  2),      # unidentified source
]

## 3. The result

In [ ]:
table = convert_many(observations)
table

name,ra,dec,fil,leff,mag,emag,av,flux,eflux,dflux,deflux
,deg,deg,,Angstrom,mag,mag,,erg / (Angstrom s cm2),erg / (Angstrom s cm2),erg / (Angstrom s cm2),erg / (Angstrom s cm2)
str6,float64,float64,str14,float64,float64,float64,float64,float64,float64,float64,float64
AR Aur,79.57,33.73,2MASS/2MASS.J,12350.0,6.19,0.019,0.0007,1.0456962802417e-12,1.8299315466798e-14,1.0459011158382e-12,1.8302900017368e-14
AR Aur,79.57,33.73,2MASS/2MASS.H,16620.0,14.6,0.12,2.0,1.6376832602551e-16,1.8100344298125e-17,2.3309283985535e-16,2.576237272006e-17
AR Aur,79.57,33.73,2MASS/2MASS.Ks,21590.0,14.3,0.14,2.0,8.1610882550366e-17,1.0523296088847e-17,1.0267750165474e-16,1.323972634298e-17
V 1234,85.12,-2.41,2MASS/2MASS.H,16620.0,12.7,0.06,--,9.4238835265933e-16,5.207830494347e-17,9.4238835265933e-16,5.207830494347e-17
--,--,--,2MASS/2MASS.J,12350.0,15.2,0.15,2.0,2.6025888397803e-16,3.5956093594024e-17,4.5545265532304e-16,6.2923109682683e-17


## 4. Saving it

In [ ]:
save_table(table, "fluxes.csv")
save_table(table, "fluxes.vot")

'fluxes.vot'